# UniChart — the `plot_marginal()` method, end to end

`nb.plot_marginal()` draws a scatter of `y` against `x` and hangs a distribution
of each variable off the matching axis: `x`'s in a strip **above** the plot,
`y`'s in a strip to its **right**. Each strip shares the main panel's axis, so
the histogram bars (or box, violin, rug, KDE) line up with the points below /
beside them. Every selected dataset is overlaid, in its own color, in both the
scatter and the strips.

| Section | What it shows |
|---|---|
| 1 | Basic call — scatter + histogram strips |
| 2 | The `plot(by='marginal')` alias |
| 3 | `marginal=` — histogram, box, violin, rug, kde |
| 4 | `marginal_x` / `marginal_y` — one kind per side, or a side switched off |
| 5 | `marginal_size` — how much of the block the strips take |
| 6 | Histogram binning — `nbins`, `bin_size` / `bin_start` / `bin_end`, `histnorm` |
| 7 | Several (x, y) pairs — the block grid, `ncols`, `subplot_titles` |
| 8 | `by='sets'` — one block per dataset |
| 9 | Styling — dataset color / marker / alpha, `color=`, `alpha=`, `hue`, `reg_order` |
| 10 | Reference lines and highlights land on the strips too |
| 11 | Legends — `legend='above' / 'right' / 'off'`, `suppress_legends` |
| 12 | Titles, footer, spacing, figure size, `set_plot_size` |
| 13 | Notebook-wide defaults via `set_default_format` |
| 14 | Argument validation |

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook


def engine_df(seed, egt_offset, ff_gain, n=250):
    '''One engine's worth of steady-state samples: fan speed N1 drives exhaust
    temperature EGT and fuel flow FF, with outside air temperature OAT as a
    second influence on EGT. Scattered samples rather than a time series, so
    the marginal distributions have something to say.'''
    r = np.random.default_rng(seed)
    N1  = np.clip(r.normal(75, 12, n), 40, 100)
    OAT = r.normal(15 + 4 * seed, 8, n)
    EGT = 300 + 4.0 * N1 + egt_offset + 0.8 * OAT + r.normal(0, 12, n)
    FF  = (200 + 30 * N1) * ff_gain + r.normal(0, 60, n)
    return pd.DataFrame({'N1': N1, 'EGT': EGT, 'FF': FF, 'OAT': OAT})


def build():
    '''A fresh notebook with three engines loaded — each section starts from
    this so the examples don't inherit each other's formatting.'''
    nb = UnichartNotebook()
    nb.load_df(engine_df(1,   0, 1.00), title='Engine A')
    nb.load_df(engine_df(2,  25, 1.08), title='Engine B')
    nb.load_df(engine_df(3, -15, 0.94), title='Engine C')
    return nb


build().list_sets()

/home/tom/Documents/GitHub/unichart/unichart.py:8638: SyntaxWarning: invalid escape sequence '\('
  snippet = """


UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


Set,Title,Selected,Shape,Query
0,Engine A,✓,250 x 4,None
1,Engine B,✓,250 x 4,None
2,Engine C,✓,250 x 4,None


---
## 1. Basic call — scatter + histogram strips

The default is `marginal='histogram'`: a histogram of `x` above the scatter and
a histogram of `y` to its right. The strips share the main panel's axes (zoom
the scatter and the strips follow), and their count axes are hidden — the
strips are there to show *shape*, not to be read off.

The scatter itself is styled exactly like `plot()`: each dataset's color,
marker, alpha, hover parameters and so on all carry over.

In [2]:
nb = build()
nb.plot_marginal(x='N1', y='EGT')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


## 2. The `plot(by='marginal')` alias

`plot_marginal` is also reachable through the main `plot()` method with
`by='marginal'`, alongside `by='ymult'`. Any extra keyword arguments
(`marginal=`, `nbins=`, …) are forwarded.

In [3]:
nb = build()
nb.plot(x='N1', y='EGT', by='marginal', marginal='box')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


## 3. `marginal=` — the five kinds

`marginal` picks the distribution drawn in **both** strips:

| kind | what you get |
|---|---|
| `'histogram'` *(default)* | binned counts, overlaid per dataset |
| `'box'` | box-and-whisker, one per dataset |
| `'violin'` | kernel-density violin, one per dataset |
| `'rug'` | one tick per sample |
| `'kde'` | a smooth Gaussian kernel-density curve (needs `scipy`) |

In [4]:
nb = build()
nb.plot_marginal(x='N1', y='EGT', marginal='violin', suptitle="marginal='violin'")

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


In [5]:
nb.plot_marginal(marginal='kde', suptitle="marginal='kde'")   # x / y are remembered from the last call

In [6]:
nb.plot_marginal(marginal='rug', suptitle="marginal='rug'")

## 4. `marginal_x` / `marginal_y` — per-side control

`marginal_x` and `marginal_y` override the kind for one side. Pass `False` to
drop that strip entirely — the main panel then grows into the freed space.

In [7]:
nb = build()
nb.plot_marginal(x='N1', y='EGT', marginal_x='kde', marginal_y='box',
                 suptitle='KDE above, box to the right')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


In [8]:
nb.plot_marginal(x='N1', y='EGT', marginal_y=False,
                 suptitle='Only the x distribution (marginal_y=False)')

## 5. `marginal_size` — the strips' share of the block

`marginal_size` is the fraction of each block given to a strip (default
`0.2`). It must be below `0.6`; the main panel takes the rest. The gap between
the main panel and its strips is a fixed few pixels regardless.

In [9]:
nb = build()
nb.plot_marginal(x='N1', y='EGT', marginal_size=0.35, suptitle='marginal_size=0.35')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


## 6. Histogram binning

With `marginal='histogram'` the same binning arguments as `histogram()` apply
to the strips: `nbins` (a target count), or an explicit `bin_size` /
`bin_start` / `bin_end`, and `histnorm` (`''` for counts, `'percent'`,
`'probability'`, `'density'`, `'probability density'`). They are ignored for
the other kinds.

In [10]:
nb = build()
nb.plot_marginal(x='N1', y='EGT', nbins=10, histnorm='percent',
                 suptitle='nbins=10, histnorm=percent')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


In [11]:
nb.plot_marginal(x='N1', y='EGT', bin_size=5, bin_start=40, bin_end=100,
                 suptitle='Explicit 5-unit bins from 40 to 100 (both strips)')

## 7. Several (x, y) pairs — the block grid

Pass a list to `y=` (or to `x=`, or both of equal length) and each pair gets its
own **block** — a main panel plus its two strips. Blocks tile a grid sized
like every other unichart grid plot (`ncols` / `nrows`, with the usual
`set_default_format(ncols=)` fallback and the sticky last grid). Each block
can be titled with `subplot_titles`.

In [12]:
nb = build()
nb.plot_marginal(x='N1', y=['EGT', 'FF', 'OAT'], ncols=2,
                 subplot_titles=['Exhaust temp', 'Fuel flow', 'Outside air'],
                 figsize=(14, 10))

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


In [13]:
# Equal-length lists pair up positionally: (N1, EGT) and (OAT, EGT).
nb.plot_marginal(x=['N1', 'OAT'], y=['EGT', 'EGT'], marginal='kde', ncols=2)

## 8. `by='sets'` — one block per dataset

`by='sets'` (or `'datasets'`) gives every selected dataset its own block for a
single `x` / `y` pair, titled with the dataset's name. Use it when the
overlaid strips get too busy to compare.

In [14]:
nb = build()
nb.plot_marginal(x='N1', y='EGT', by='sets', ncols=3, figsize=(16, 6))

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


In [15]:
# Selection is honoured as everywhere else.
nb.select([0, 2])
nb.plot_marginal(x='N1', y='EGT', by='sets', marginal='violin')

## 9. Styling

The scatter takes each dataset's own styling (`color`, `marker`, `markersize`,
`alpha`, …); the strips take the dataset's color.

- `color=` forces one color on every dataset (scatter and strips).
- `alpha=` sets the strips' opacity (default `0.7`); the scatter keeps each
  dataset's own alpha.
- `hue` colors the *points* by a column; the strips keep the flat dataset
  color, since a distribution has no per-point identity to carry a hue.
- `reg_order` draws the fitted trendline on the main panel, as in `plot()`.

In [16]:
nb = build()
nb.marker(0, 'square'); nb.marker(1, 'diamond'); nb.marker(2, 'triangle-up')
nb.alpha('all', 0.5)
nb.plot_marginal(x='N1', y='EGT', suptitle='Per-dataset markers and alpha carry over')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


In [17]:
nb = build()
nb.plot_marginal(x='N1', y='EGT', color='slategray', alpha=0.3,
                 suptitle="color='slategray', alpha=0.3 for the strips")

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


In [18]:
nb = build()
nb.select([1])
nb.hue(1, 'OAT')                      # points colored by outside air temperature
nb.plot_marginal(x='N1', y='EGT', marginal='kde',
                 suptitle='hue colors the points; the strips keep the set color')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


In [19]:
nb = build()
nb.reg_order('all', 1)
nb.plot_marginal(x='N1', y='EGT', marginal='box', suptitle='Linear fits per dataset')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


## 10. Reference lines and highlights

`nb.line()` and `nb.highlight()` decorate marginal plots too. A decoration on
the `x` variable is drawn on the main panel **and** the top strip; one on the
`y` variable on the main panel **and** the right strip — so the threshold you
mark on the scatter is visible against the distribution as well.

In [20]:
nb = build()
nb.line('N1', 90, color='red', dash='dash', label='N1 limit')
nb.highlight('EGT', (640, 700), color='orange', alpha=0.25)
nb.plot_marginal(x='N1', y='EGT', suptitle='x line -> main + top strip; y highlight -> main + right strip')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


## 11. Legends

`legend` matches `plot()`: `'above'` (default, horizontal, below the
suptitle), `'right'` (vertical) or `'off'`. `suppress_legends=True` keeps the
legend but hides the entries. Both fall back to the `set_default_format`
defaults when not passed. Only the scatter traces carry legend entries; the
strips are grouped under them.

In [21]:
nb = build()
nb.plot_marginal(x='N1', y='EGT', legend='right', suptitle="legend='right'")

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


In [22]:
nb.plot_marginal(x='N1', y='EGT', legend='off', suptitle="legend='off'")

## 12. Titles, footer, spacing and size

`suptitle` and `footer` behave as in every other method (a stored
`nb.suptitle` is the fallback). `hspace` / `vspace` set the gap **between
blocks** — pixels if 1 or more, a fraction of the plot area if below 1. The
gap between a main panel and its strips is not affected.

`set_plot_size` pins the size of the **main panel** of each block (in inches,
like `figsize`); the strips are added on top of that.

In [23]:
nb = build()
nb.plot_marginal(x='N1', y=['EGT', 'FF'], ncols=2, hspace=120,
                 suptitle='Two blocks, 120 px apart',
                 footer='Synthetic engine data — three engines, 250 samples each',
                 figsize=(14, 7))

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


In [24]:
nb = build()
nb.set_plot_size(4, 4)        # 4 in x 4 in main panel per block
fig = nb.plot_marginal(x='N1', y='EGT', suptitle='set_plot_size(4, 4) pins the main panel')
nb.set_plot_size(reset=True)
fig

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


## 13. Notebook-wide defaults

`set_default_format` seeds the arguments `plot_marginal` shares with the other
methods whenever a call doesn't pass its own value: `histnorm` and `alpha`
(shared with `histogram`), `legend` and `suppress_legends` (shared with
`plot`), `ncols` / `nrows`, `hspace` / `vspace` and `figsize`. An explicit
argument always wins.

In [25]:
nb = build()
nb.set_default_format(histnorm='probability density', alpha=0.4, legend='right')
nb.plot_marginal(x='N1', y='EGT', suptitle='Defaults: density-normalised, alpha 0.4, legend right')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B


Loaded Set 2: Engine C


In [26]:
nb.plot_marginal(x='N1', y='EGT', legend='above', suptitle='Per-call legend=above overrides the default')

## 14. Argument validation

Bad inputs fail before anything is drawn:

- `marginal` (or a per-side override) must be one of the five kinds or `False`;
- `marginal_size` must be strictly between 0 and 0.6;
- `by='sets'` takes a single `x` and a single `y`.

In [27]:
nb = build()
for kwargs in [dict(marginal='hist'),
               dict(marginal_size=0.75),
               dict(by='sets', y=['EGT', 'FF'])]:
    try:
        nb.plot_marginal(x='N1', y=kwargs.pop('y', 'EGT'), **kwargs)
    except ValueError as e:
        print(f'{kwargs} -> ValueError: {e}')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C
{'marginal': 'hist'} -> ValueError: marginal must be one of ('histogram', 'box', 'violin', 'rug', 'kde') or False, got 'hist'
{'marginal_size': 0.75} -> ValueError: marginal_size must be between 0 and 0.6, got 0.75
{'by': 'sets'} -> ValueError: plot_marginal(by='sets') takes a single y variable
